# p-Hacking & the Multiple-Comparisons Trap

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/lectures/p_hacking.ipynb)

Test enough signals and some will look brilliant **by pure chance**. This is the single most
important reason backtests lie — and exactly what the ConvexPi Lab's out-of-sample grading is built
to catch. We'll manufacture the illusion, measure it, and then fix it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(7)
print("ready")

## 1. Manufacture the illusion

We have one asset's monthly returns — **pure noise**, no predictability. We also invent **200
candidate signals**, also pure noise, unrelated to returns. None *should* predict anything.

In [ ]:
T, K = 240, 200                      # 240 months, 200 candidate signals
returns = rng.normal(0, 0.05, T)     # the target: pure noise
signals = rng.normal(0, 1, (T, K))   # 200 candidate predictors: also pure noise

# Correlate each signal with next-month return; record t-stat and p-value.
tstats, pvals = [], []
for k in range(K):
    r = np.corrcoef(signals[:-1, k], returns[1:])[0, 1]
    t = r * np.sqrt((T - 2) / (1 - r**2))
    tstats.append(t); pvals.append(2 * (1 - stats.t.cdf(abs(t), T - 2)))
tstats, pvals = np.array(tstats), np.array(pvals)

sig5 = (pvals < 0.05).sum()
print(f"signals 'significant' at p<0.05: {sig5} of {K}  ({100*sig5/K:.0f}%)")
print(f"best |t-stat|: {np.abs(tstats).max():.2f}  (looks impressive — and is 100% luck)")

About **5% show up 'significant' at p<0.05** — exactly the false-positive rate you'd expect from
random noise. Cherry-pick the best and you have a 'strategy' with a great t-stat and zero substance.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].hist(tstats, bins=30, color='steelblue'); ax[0].set_title('t-stats of 200 noise signals'); ax[0].axvline(0, color='grey', lw=0.5)
ax[1].hist(pvals, bins=20, color='indianred'); ax[1].set_title('p-values (uniform = no signal)'); ax[1].axvline(0.05, color='k', ls='--', lw=1)
plt.tight_layout(); plt.show()

## 2. The fix: correct for how many things you tried

The p-value of your *best* signal must account for the fact that you ran **K** tests. Two standard
corrections:

- **Bonferroni**: require p < 0.05 / K (controls the chance of *any* false positive).
- **Benjamini-Hochberg (FDR)**: controls the *expected fraction* of false positives among discoveries.

In [ ]:
best = pvals.min()
print(f"best raw p-value      : {best:.4f}  -> 'significant'")
print(f"Bonferroni threshold  : {0.05/K:.5f}")
print(f"best p after Bonferroni: {min(best*K,1):.3f}  -> {'survives' if best*K < 0.05 else 'NOT significant'}")

# Benjamini-Hochberg: how many discoveries survive FDR control at 5%?
order = np.argsort(pvals); ranked = pvals[order]
bh = ranked <= (np.arange(1, K+1)/K)*0.05
n_bh = np.where(bh)[0].max()+1 if bh.any() else 0
print(f"BH-FDR discoveries at 5%: {n_bh}  (correct answer: 0)")

## 3. The real test: out of sample

Corrections help, but the **honest** check is to evaluate the winner on data you never searched.
Pick the best in-sample signal, then score it on a fresh, held-out stretch.

In [ ]:
half = T // 2
is_t = []
for k in range(K):
    r = np.corrcoef(signals[:half-1, k], returns[1:half])[0, 1]
    is_t.append(abs(r*np.sqrt((half-2)/(1-r**2))))
winner = int(np.argmax(is_t))

def corr(a, b): return np.corrcoef(a, b)[0, 1]
ic_is  = corr(signals[:half-1, winner],  returns[1:half])
ic_oos = corr(signals[half:-1, winner], returns[half+1:])
print(f"winner signal #{winner}")
print(f"  in-sample correlation : {ic_is:+.3f}  (selected to be large)")
print(f"  out-of-sample         : {ic_oos:+.3f}  (back to ~0 — the edge was never real)")

## Takeaways

1. **Searching many signals inflates false positives.** ~5% of pure-noise signals "work" at p<0.05.
2. **Correct for the search** (Bonferroni / FDR) — your effective p-value depends on how many things you tried.
3. **Out-of-sample is the ultimate judge.** A signal selected for in-sample fit reverts to noise OOS.

This is why the ConvexPi Lab grades your strategy on **hidden** data and reports an **overfitting
ratio** — a beautiful in-sample Sharpe that collapses out of sample is the tell. → Try Mission 1.